## Converting .CSV Volumes into Delta Tables

The cell below loops through a list of CSV files (`categories`, `cities`, `countries`, `customers`, `employees`, `products`, `sales`) stored in the `grocerydata_csvs` Unity Catalog Volume. For each file it:

1. Reads the CSV with `spark.read`, using the first row as a header and inferring column data types.
2. Cleans up the column names — stripping whitespace, lowercasing, and replacing spaces with underscores — so they follow standard Delta table naming conventions.
3. Writes each DataFrame as a Delta table (`saveAsTable`) into the `grocery_sales.default` schema, overwriting any existing table with the same name.

The end result is seven managed Delta tables in Unity Catalog, one per source CSV file.

In [ ]:
base = "/Volumes/grocery_sales/default/grocerydata_csvs/"
files = ["categories","cities","countries","customers","employees","products","sales"]

for name in files:
    df = (spark.read
          .option("header", "true")
          .option("inferSchema", "true")
          .csv(f"{base}{name}.csv"))
    
    df = df.toDF(*[c.strip().lower().replace(" ", "_") for c in df.columns])

    df.write.mode('overwrite').saveAsTable(f'grocery_sales.default.{name}')